# Kaggle llama.cpp backend runner

Notebook pendek. Logic panjang ada di repo.

## Cell 1 — Dependency + prebuilt llama.cpp CUDA

In [ ]:
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/kaggle-llamacpp-backend.git"
REPO_DIR = Path("/kaggle/working/kaggle-llamacpp-backend")

if not REPO_DIR.exists():
    if "YOUR_USERNAME" in REPO_URL:
        raise ValueError("Ubah REPO_URL dulu ke repo GitHub kamu.")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))

from kaggle_llamacpp import ensure_llamacpp_cuda

state = ensure_llamacpp_cuda()
state


## Cell 2 — Download model GGUF

Downloader: aria2c + tqdm.

In [ ]:
from kaggle_llamacpp import download_model

MODEL_URL = "https://huggingface.co/mradermacher/gemma-4-26B-A4B-Heretic-Stable-i1-GGUF/resolve/main/gemma-4-26B-A4B-Heretic-Stable.i1-Q4_K_M.gguf"

model_path = download_model(MODEL_URL, backend="aria2")
model_path


## Cell 3 — Config + start llama-server + health check + response test

In [ ]:
from kaggle_llamacpp import ServerConfig, start_llama_server, wait_until_ready, test_chat_completion

cfg = ServerConfig(
    ctx_size=2048,
    batch_size=256,
    ubatch_size=64,
    port=8080,
    gpu_layers=999,
    split_mode="layer",
    tensor_split="1,1",
    threads=4,
    parallel=1,
    reasoning_format="none",
    enable_thinking=False,
)

pid = start_llama_server(cfg)

ready = wait_until_ready(port=cfg.port, timeout_s=900)
if not ready:
    raise RuntimeError("Server belum ready atau proses mati.")

text = test_chat_completion(
    "Say hello in one short sentence. Do not explain your reasoning.",
    port=cfg.port,
    max_tokens=64,
)
print("FINAL:", text)


## Cell 4 — Start ngrok tunnel

Buat Kaggle Secret `NGROK_AUTHTOKEN` dulu.

In [ ]:
from kaggle_llamacpp import start_ngrok_tunnel

public_url = start_ngrok_tunnel(
    port=8080,
    secret_name="NGROK_AUTHTOKEN",
)

print("OpenAI-compatible Base URL:")
print(public_url + "/v1")


## Optional — status dan cleanup

In [ ]:
from kaggle_llamacpp import print_status
print_status(lines=120)


In [ ]:
from kaggle_llamacpp import stop_llama_server, stop_ngrok_tunnel

# stop_ngrok_tunnel()
# stop_llama_server()
